# Convolutional flows for periodic 1D signals
`ConvFlow1d` models tensors with shape `(batch, channels, length)`. This notebook compares alternating affine coupling blocks with Glow blocks, which add learned invertible channel mixing and ActNorm. It also demonstrates circular padding, temperature-controlled sampling, and portable checkpoints.

## Setup
Install the current checkout with `python -m pip install -e '.[examples]'` from the repository root. These examples exercise recent source APIs, which may not yet be in an older installed release. Select that environment's kernel and run all cells in order.

All observations are synthetic; no download is required. The small training budgets demonstrate the API rather than a converged scientific model. Increase `steps` and model capacity for a longer experiment.

In [ ]:
import torch
import antsnormflows as nf
from matplotlib import pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Package:", nf.__version__, "Device:", device)


## Generate noisy periodic signals
The two channels share a random phase. Circular padding is appropriate because the last time point neighbors the first; it is not generally appropriate for an open-ended time series. We exclude the repeated endpoint and add observation noise so the data have a full-dimensional density.

In [ ]:
channels, length = 2, 32
time = torch.arange(length, device=device) * (2 * torch.pi / length)
def draw_signals(batch):
    phase = 2 * torch.pi * torch.rand(batch, 1, device=device)
    amplitude = 0.8 + 0.4 * torch.rand(batch, 1, device=device)
    signal = torch.stack([amplitude * torch.sin(time + phase),
                          amplitude * torch.cos(time + phase)], dim=1)
    return signal + 0.15 * torch.randn_like(signal)

observations = draw_signals(64)
plt.plot(time.cpu(), observations[0].T.cpu())
plt.xlabel("Phase (radians)")
plt.ylabel("Amplitude")
plt.show()


## Train both architectures
`actnorm=True` adds normalization to the coupling-only model. Glow blocks already contain it. Scale and shift caps constrain learned transformations; inputs should still be sensibly scaled. Initialize data-dependent normalization on observations before generating samples.

In [ ]:
def make_model(use_glow):
    return nf.ConvFlow1d(
        channels=channels, length=length, K=3, hidden_channels=16,
        kernel_size=3, padding_mode="circular", use_glow_blocks=use_glow,
        actnorm=True, scale_cap=0.5, shift_cap=3.0,
        actnorm_s_cap=5.0, conv_s_cap=1.0,
    ).to(device)

steps = 50
models, histories = {}, {}
validation = draw_signals(64)
for name, use_glow in [("coupling", False), ("glow", True)]:
    model = make_model(use_glow)
    with torch.no_grad():
        model.log_prob(observations)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    history = []
    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = model.forward_kld(draw_signals(32))
        assert torch.isfinite(loss)
        loss.backward()
        optimizer.step()
        history.append(loss.item() / (channels * length))
    model.eval()
    with torch.no_grad():
        print(name, "validation NLL per scalar:",
              -model.log_prob(validation).mean().item() / (channels * length))
    models[name], histories[name] = model, history

for name, history in histories.items():
    plt.plot(history, label=name)
plt.ylabel("Training NLL per scalar (nats)")
plt.xlabel("Step")
plt.legend()
plt.show()


## Verify invertibility and sample at different temperatures
At temperature 1, sampled log densities should agree with `log_prob`. At other temperatures the returned log density describes the tempered distribution, not the ordinary training density. `ConvFlow1d.sample` restores the previous base temperature after sampling.

In [ ]:
model = models["glow"]
with torch.no_grad():
    z, inverse_ld = model.inverse_and_log_det(validation)
    reconstructed, forward_ld = model.forward_and_log_det(z)
    torch.testing.assert_close(reconstructed, validation, atol=2e-4, rtol=2e-4)
    torch.testing.assert_close(inverse_ld + forward_ld,
                               torch.zeros_like(inverse_ld), atol=2e-3, rtol=0)
    samples, log_q = model.sample(8)
    torch.testing.assert_close(log_q, model.log_prob(samples), atol=2e-3, rtol=2e-4)

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
previous_temperature = model.q0.temperature
for ax, temperature in zip(axes, [0.7, 1.0, 1.3]):
    samples, _ = model.sample(4, temperature=temperature)
    ax.plot(time.cpu(), samples[0].T.cpu())
    ax.set_title(f"Temperature {temperature}")
    ax.set_xlabel("Phase")
assert model.q0.temperature == previous_temperature
plt.tight_layout()
plt.show()


## Reload a checkpoint
The architecture and its constructor options must be retained separately from `state_dict`. A temporary directory keeps the example from overwriting an existing checkpoint; use your own output directory for a persistent result.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "conv_flow_1d.pt"
    model.save(path)
    restored = make_model(True)
    restored.load(path, map_location=device)
    restored.eval()
    with torch.no_grad():
        torch.testing.assert_close(restored.log_prob(validation), model.log_prob(validation))
print("Checkpoint round trip passed.")
